In [1]:
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.decomposition import LatentDirichletAllocation
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score

print("Environment setup successful.")

Environment setup successful.


In [2]:
documents = [
    # Formula 1 / Motorsports (Class 0)
    "Ferrari introduced a new front wing upgrade to increase aerodynamic downforce and cornering speed",
    "Max Verstappen set the fastest lap during qualifying session using soft compound tires",
    "Pit stop strategy and tire degradation determine race victory in high temperature grand prix conditions",
    "Mercedes adjusted suspension settings to reduce porpoising on high speed street circuits",
    "Engine reliability and turbocharger efficiency are critical factors for championship success",

    # Space / Astronomy (Class 1)
    "James Webb telescope captured deep space images of distant galaxies and newborn stars",
    "NASA launched a new rover to explore Martian surface soil for organic compounds",
    "Astronomers observed a massive black hole pulling interstellar gas from nearby solar systems",
    "SpaceX rocket completed orbit insertion carrying satellite payloads into low Earth orbit",
    "Lunar exploration missions aim to find water ice inside shadowed craters near the south pole",

    # Finance / Markets (Class 2)
    "Central banks raised interest rates to control inflation and stabilize national currency values",
    "Stock market indices surged following quarterly earnings reports from major tech companies",
    "Investors diversified portfolios into corporate bonds treasury bills and market commodities",
    "Venture capital firms allocated seed funding to artificial intelligence and fintech startups",
    "Global economic growth projections were adjusted lower due to supply chain disruptions"
]

labels = [0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 2, 2, 2, 2, 2]
class_names = ["Motorsports", "Space", "Finance"]

print(f"Loaded {len(documents)} documents across {len(class_names)} categories.")

Loaded 15 documents across 3 categories.


In [5]:
# Cell 3: Feature Extraction and LDA Topic Modeling

# Step 1: Convert raw text to word count vectors
vectorizer = CountVectorizer(stop_words='english', min_df=1)
X_counts = vectorizer.fit_transform(documents)

# Step 2: Fit LDA model using Gibbs sampling / online Bayes inference
# n_components matches our expected number of topics (3)
num_topics = 3
lda_model = LatentDirichletAllocation(
    n_components=num_topics,
    learning_method='online',  # Stochastic variational / Gibbs approach
    random_state=42
)

# Transform documents into topic probability distributions (N x num_topics)
X_topics = lda_model.fit_transform(X_counts)

print("--- Document-Topic Matrix (First 3 Documents) ---")
print(X_topics[:3])

--- Document-Topic Matrix (First 3 Documents) ---
[[0.03082677 0.93804607 0.03112716]
 [0.02837021 0.02846888 0.9431609 ]
 [0.0243171  0.02435429 0.95132861]]


In [8]:
# Cell 4: Top words per LDA topic
feature_names = vectorizer.get_feature_names_out()

print("--- Discovered Topics & Top Words ---")
for topic_idx, topic_weights in enumerate(lda_model.components_):
    top_word_indices = topic_weights.argsort()[:-6:-1]
    top_words = [feature_names[i] for i in top_word_indices]
    print(f"Topic {topic_idx + 1}: {', '.join(top_words)}")

--- Discovered Topics & Top Words ---
Topic 1: orbit, supply, projections, tech, carrying
Topic 2: new, launched, speed, nasa, capital
Topic 3: high, grand, max, verstappen, prix


In [9]:
# Cell 5: Train and Evaluate Classifier

# Split data into training and test sets
X_train, X_test, y_train, y_test = train_test_split(
    X_topics, labels, test_size=0.33, random_state=42, stratify=labels
)

# Train Logistic Regression on topic distributions
classifier = LogisticRegression(random_state=42)
classifier.fit(X_train, y_train)

# Predict on test set
y_pred = classifier.predict(X_test)

# Display evaluation metrics
print("--- Classification Results ---")
print(f"Accuracy: {accuracy_score(y_test, y_pred) * 100:.2f}%\n")
print(classification_report(y_test, y_pred, target_names=class_names))

--- Classification Results ---
Accuracy: 40.00%

              precision    recall  f1-score   support

 Motorsports       0.33      1.00      0.50         1
       Space       0.50      0.50      0.50         2
     Finance       0.00      0.00      0.00         2

    accuracy                           0.40         5
   macro avg       0.28      0.50      0.33         5
weighted avg       0.27      0.40      0.30         5



/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
